# Hyperparameter Selection — Cross-Sectional Split

## Design

Split is **cross-sectional**: 20% of stocks for tuning, 80% for estimation.  
Each set uses the full time series (1,879 days) — no window-length problem.

## Optuna search (fast, single-stock trials)

Each trial evaluates (λ, s) on **one randomly chosen tuning stock**.  
This is fast (seconds per trial) because features are globally standardised:
selection rate is primarily a function of (λ, window) not of which stock.  
After optimisation, we validate stability on a 30-stock subsample.

## Objective (CMCE-aligned)

| Component | Weight | Rationale |
|---|---|---|
| σ(κ_tstat − 1.96) | 0.7 | κ identification |
| exp(−((sel_rate − 0.03)/0.02)²) | 0.2 | Bell-shaped: peak at 3% sparsity |
| σ((κ − 0.3)/0.2) | 0.1 | Interior prior |

Hard constraints: sel_rate < 0.003 or sel_rate > 0.20 → score = 0  
(Below 0.3%: f_t degenerate; above 20%: dense regime, κ → 0)


In [ ]:
import sys, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
from joblib import Parallel, delayed

optuna.logging.set_verbosity(optuna.logging.WARNING)

sys.path.insert(0, '../scripts')
from grid_search import estimate_single_config_fast

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

TUNE_FRAC       = 0.20   # fraction of stocks for tuning
N_LAGS          = 1      # fixed
N_TRIALS        = 200    # Optuna trials — fast because 1 stock each
N_VALIDATE      = 30     # stocks for post-hoc stability check

# λ range: base lambda; actual per-stock λ = λ_base × (y_vol / TARGET_VOL)
# This makes selection rate vol-invariant (X is standardised but y is not).
# Median daily return vol across 1620 stocks is ~1.75%.
TARGET_VOL   = 0.0175   # reference volatility for λ scaling
LAMBDA_LOW,  LAMBDA_HIGH = 1e-3, 0.1
WINDOW_LOW,  WINDOW_HIGH = 60,   400

print('Setup complete.')


In [ ]:
X_all = pd.read_csv('../../Data/clean_data/final_macro_topic_features.csv',
                    index_col=0, parse_dates=True)
R_all = pd.read_csv('../../Data/data_raw/cross_sectional_returns.csv',
                    index_col=0, parse_dates=True)

common_idx = X_all.index.intersection(R_all.index)
X_all = X_all.loc[common_idx]
R_all = R_all.loc[common_idx]
print(f'Features : {X_all.shape}')
print(f'Returns  : {R_all.shape}')
print(f'Common   : {len(common_idx)} days  [{common_idx[0].date()} – {common_idx[-1].date()}]')

In [ ]:
min_obs = WINDOW_HIGH + N_LAGS + 100
eligible = [st for st in R_all.columns
            if R_all[st].dropna().shape[0] >= min_obs]
print(f'Eligible stocks (≥{min_obs} obs): {len(eligible)}')

rng = np.random.default_rng(SEED)
perm = list(rng.permutation(len(eligible)))
eligible = [eligible[i] for i in perm]

n_tune = max(1, int(len(eligible) * TUNE_FRAC))
TUNE_STOCKS  = eligible[:n_tune]
ESTIM_STOCKS = eligible[n_tune:]

print(f'Tuning stocks    : {len(TUNE_STOCKS)}')
print(f'Estimation stocks: {len(ESTIM_STOCKS)}')

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -50, 50)))


def score_from_summary(sm):
    """
    CMCE-aligned score.
    Hard floor/ceiling on selection rate; bell-shaped sparsity reward.
    """
    if sm is None:
        return np.nan
    tstat    = float(sm.get('kappa_tstat',        np.nan))
    sel_rate = float(sm.get('avg_selection_rate', np.nan))
    kappa    = float(sm.get('kappa',              np.nan))
    if not all(np.isfinite([tstat, sel_rate, kappa])):
        return np.nan
    if sel_rate < 0.003:  # f_t degenerate (zero selection)
        return 0.0
    if sel_rate > 0.20:   # too dense: κ → 0
        return 0.0
    # Gaussian bell: peak at sel_rate=3%, half-max at ~5.8%
    sel_reward   = np.exp(-((sel_rate - 0.03) / 0.02) ** 2)
    tstat_reward = sigmoid(tstat - 1.96)
    kappa_reward = sigmoid((kappa - 0.3) / 0.2)
    return 0.7 * tstat_reward + 0.2 * sel_reward + 0.1 * kappa_reward


def run_one(st, window_size, lam_base, n_lags=1):
    """Estimate one stock. Scales lambda by stock vol so selection rate
    is approximately vol-invariant (X is standardised, y is not)."""
    r = R_all[st].dropna()
    if len(r) < window_size + n_lags + 100:
        return None
    # vol-adjusted lambda
    stock_vol = float(r.std())
    eff_lam   = lam_base * (stock_vol / TARGET_VOL)
    X = X_all.loc[r.index]
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        try:
            res = estimate_single_config_fast(
                X=X, y=r, window_size=window_size,
                n_lags=n_lags, lambda_val=eff_lam, return_details=False
            )
            sm = res.get('summary', None)
            if sm is not None:
                sm['eff_lambda'] = eff_lam   # store for diagnostics
            return sm
        except Exception:
            return None


print('Helper functions defined (vol-adjusted lambda).')


## Optuna Search on Tuning Stocks

Each trial draws a fresh random subsample of `N_STOCKS_TRIAL` tuning stocks and evaluates
the mean composite score.  Using a subsample per trial keeps each trial fast while sampling
broadly from the tuning population.

In [ ]:
# Pre-assign one stock per trial (reproducible)
trial_rng = np.random.default_rng(SEED + 1)
trial_stocks = list(
    trial_rng.choice(TUNE_STOCKS, size=N_TRIALS, replace=True)
)


def objective(trial):
    lam = trial.suggest_float('lambda',      LAMBDA_LOW, LAMBDA_HIGH, log=True)
    s   = trial.suggest_int(  'window_size', WINDOW_LOW, WINDOW_HIGH)
    st  = trial_stocks[trial.number]
    sm  = run_one(st, s, lam, N_LAGS)
    sc  = score_from_summary(sm)
    return float(sc) if (sc is not None and np.isfinite(sc)) else 0.0


sampler = optuna.samplers.TPESampler(seed=SEED)
study   = optuna.create_study(direction='maximize', sampler=sampler,
                               study_name='cs_tuning_single_stock')

print(f'Running {N_TRIALS} Optuna trials (1 stock each) …')
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best = study.best_trial
print(f'\nBest → lambda={best.params["lambda"]:.3e}, '
      f'window={best.params["window_size"]}, score={best.value:.4f}')


In [ ]:
df_trials = study.trials_dataframe()
df_trials = df_trials[df_trials['value'].notna()].copy()
df_trials['log_lambda'] = np.log10(df_trials['params_lambda'])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sc = axes[0].scatter(df_trials['log_lambda'], df_trials['params_window_size'],
                     c=df_trials['value'], cmap='viridis', s=40)
plt.colorbar(sc, ax=axes[0], label='Score')
axes[0].set_xlabel('log₁₀(λ)'); axes[0].set_ylabel('window')
axes[0].set_title('Optuna landscape')
best_row = df_trials.loc[df_trials['value'].idxmax()]
axes[0].scatter(best_row['log_lambda'], best_row['params_window_size'],
                color='red', s=150, marker='*', zorder=5, label='best')
axes[0].legend()

axes[1].scatter(df_trials['log_lambda'], df_trials['value'], alpha=0.5, s=25)
axes[1].set_xlabel('log₁₀(λ)'); axes[1].set_ylabel('Score')
axes[1].set_title('Score vs λ')

axes[2].scatter(df_trials['params_window_size'], df_trials['value'], alpha=0.5, s=25)
axes[2].set_xlabel('window'); axes[2].set_ylabel('Score')
axes[2].set_title('Score vs window')

plt.tight_layout()
plt.savefig('optuna_landscape.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved optuna_landscape.png')

In [ ]:
BEST_LAMBDA = float(best.params['lambda'])
BEST_WINDOW = int(best.params['window_size'])

print('Selected hyperparameters:')
print(f'  lambda      = {BEST_LAMBDA:.6e}')
print(f'  window_size = {BEST_WINDOW}')
print(f'  n_lags      = {N_LAGS}  (fixed)')
print(f'  Optuna score = {best.value:.4f}')

# ── Stability validation on N_VALIDATE tuning stocks ─────────────────────
print(f'\nValidating on {N_VALIDATE} random tuning stocks …')
val_rng    = np.random.default_rng(SEED + 99)
val_stocks = list(val_rng.choice(TUNE_STOCKS,
                                  size=min(N_VALIDATE, len(TUNE_STOCKS)),
                                  replace=False))

val_summaries = Parallel(n_jobs=-1, backend='loky')(
    delayed(run_one)(st, BEST_WINDOW, BEST_LAMBDA, N_LAGS) for st in val_stocks
)
df_val = pd.DataFrame([sm for sm in val_summaries if sm is not None])

print(f'  stocks valid      : {len(df_val)}')
print(f'  mean sel_rate     : {df_val.avg_selection_rate.mean():.4f}  '
      f'(std={df_val.avg_selection_rate.std():.4f})')
print(f'  mean κ̂ t-stat     : {df_val.kappa_tstat.mean():.3f}  '
      f'(std={df_val.kappa_tstat.std():.3f})')
print(f'  mean κ̂            : {df_val.kappa.mean():.4f}')
print(f'  % sig κ (t>1.96)  : {(df_val.kappa_tstat > 1.96).mean()*100:.1f}%')

# Sensitivity: perturb best params slightly
perturb_rows = []
for dl in [0.7, 1.0, 1.4]:
    for ds in [-40, 0, 40]:
        lam_p = BEST_LAMBDA * dl
        s_p   = max(WINDOW_LOW, BEST_WINDOW + ds)
        scores = [score_from_summary(run_one(st, s_p, lam_p, N_LAGS))
                  for st in val_stocks[:10]]  # quick: 10 stocks
        perturb_rows.append({'lambda_factor': dl, 'window_delta': ds,
                             'score': np.nanmean([s for s in scores if s is not None])})

df_sens = pd.DataFrame(perturb_rows)
print('\nSensitivity (10-stock score):')
print(df_sens.pivot_table(values='score', index='lambda_factor',
                           columns='window_delta').to_string())


In [ ]:
# ── full estimation on ALL tuning stocks ────────────────────────────────────
print(f'Running full estimation on {len(TUNE_STOCKS)} tuning stocks …')
tune_summaries = Parallel(n_jobs=-1, backend='loky')(
    delayed(run_one)(st, BEST_WINDOW, BEST_LAMBDA, N_LAGS) for st in TUNE_STOCKS
)
df_tune = pd.DataFrame([
    {'stock': st, **sm}
    for st, sm in zip(TUNE_STOCKS, tune_summaries) if sm is not None
])

sig_t = df_tune['kappa_tstat'] > 1.96
print(f'Tuning stocks valid: {len(df_tune)}  |  sig κ: {sig_t.sum()} ({sig_t.mean()*100:.1f}%)')
print(f'mean sel_rate      : {df_tune.avg_selection_rate.mean():.4f}')
print(f'mean Stage-1 in-R² : {df_tune.r2_insample_stage1.mean():.4f}')
print(f'mean Stage-1 OOS R²: {df_tune.r2_oos_stage1.mean():.4f}')
print(f'mean Stage-2 OOS R²: {df_tune.r2_oos_stage2.mean():.4f}')
print(f'mean κ̂             : {df_tune.kappa.mean():.4f}')
print(f'mean κ̂ t-stat      : {df_tune.kappa_tstat.mean():.3f}')

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle('Tuning stocks — full estimation results', fontsize=12)

axes[0].hist(df_tune['kappa_tstat'].dropna(), bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(1.96, color='red', ls='--', lw=1.5, label='1.96')
axes[0].set_xlabel('κ t-stat'); axes[0].set_ylabel('count')
axes[0].set_title('κ t-statistics'); axes[0].legend()

axes[1].hist(df_tune['kappa'].dropna().clip(0, 1), bins=40, color='darkgreen', edgecolor='white')
axes[1].set_xlabel('κ̂'); axes[1].set_ylabel('count')
axes[1].set_title('κ̂ distribution')

axes[2].hist(df_tune['avg_selection_rate'].dropna(), bins=40, color='firebrick', edgecolor='white')
axes[2].axvline(0.03, color='k', ls='--', lw=1.5, label='target 3%')
axes[2].set_xlabel('selection rate'); axes[2].set_ylabel('count')
axes[2].set_title('LASSO selection rates'); axes[2].legend()

plt.tight_layout()
plt.savefig('tuning_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved tuning_results.png')


In [ ]:
out_path = '../../Data/clean_data/best_hyperparameters_holdout.txt'
with open(out_path, 'w') as f:
    f.write(f'window_size: {BEST_WINDOW}\n')
    f.write(f'n_lags: {N_LAGS}\n')
    f.write(f'lambda: {BEST_LAMBDA}\n')
    f.write(f'tune_frac: {TUNE_FRAC}\n')
    f.write(f'n_tune_stocks: {len(TUNE_STOCKS)}\n')
    f.write(f'n_estim_stocks: {len(ESTIM_STOCKS)}\n')
    f.write(f'optuna_score: {best.value:.6f}\n')
print(f'Saved → {out_path}')

pd.Series(TUNE_STOCKS,  name='stock').to_csv('tune_stocks.csv',  index=False)
pd.Series(ESTIM_STOCKS, name='stock').to_csv('estim_stocks.csv', index=False)
study.trials_dataframe().to_csv('optuna_cs_trials.csv', index=False)
print('Saved tune_stocks.csv, estim_stocks.csv, optuna_cs_trials.csv')

## Estimation Results on Held-Out Stocks

Full Stage-1 + Stage-2 on the 80% estimation stocks with frozen (λ*, s*).

In [ ]:
print(f'Running full estimation on {len(ESTIM_STOCKS)} estimation stocks …')
estim_summaries = Parallel(n_jobs=-1, backend='loky')(
    delayed(run_one)(st, BEST_WINDOW, BEST_LAMBDA, N_LAGS) for st in ESTIM_STOCKS
)
df_estim = pd.DataFrame([
    {'stock': st, **sm}
    for st, sm in zip(ESTIM_STOCKS, estim_summaries) if sm is not None
])

sig_e = df_estim['kappa_tstat'] > 1.96
print(f'\nEstimation stocks valid: {len(df_estim)}  |  sig κ: {sig_e.sum()} ({sig_e.mean()*100:.1f}%)')
print(f'mean sel_rate      : {df_estim.avg_selection_rate.mean():.4f}')
print(f'mean Stage-1 in-R² : {df_estim.r2_insample_stage1.mean():.4f}')
print(f'mean Stage-1 OOS R²: {df_estim.r2_oos_stage1.mean():.4f}')
print(f'mean Stage-2 OOS R²: {df_estim.r2_oos_stage2.mean():.4f}')
print(f'mean κ̂             : {df_estim.kappa.mean():.4f}')
print(f'mean κ̂ t-stat      : {df_estim.kappa_tstat.mean():.3f}')
if sig_e.any():
    print(f'\nSignificant stocks only:')
    df_s = df_estim[sig_e]
    print(f'  mean κ̂        : {df_s.kappa.mean():.4f}')
    print(f'  mean κ̂ t-stat : {df_s.kappa_tstat.mean():.3f}')
    print(f'  mean S2 OOS R²: {df_s.r2_oos_stage2.mean():.4f}')

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle('Estimation stocks — out-of-sample results', fontsize=12)

axes[0].hist(df_estim['kappa_tstat'].dropna(), bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(1.96, color='red', ls='--', lw=1.5, label='1.96')
axes[0].set_xlabel('κ t-stat'); axes[0].set_ylabel('count')
axes[0].set_title('κ t-statistics (estimation stocks)'); axes[0].legend()

axes[1].hist(df_estim['kappa'].dropna().clip(0, 1), bins=40, color='darkgreen', edgecolor='white')
axes[1].set_xlabel('κ̂'); axes[1].set_ylabel('count')
axes[1].set_title('κ̂ distribution (estimation stocks)')

axes[2].hist(df_estim['avg_selection_rate'].dropna(), bins=40, color='firebrick', edgecolor='white')
axes[2].axvline(0.05, color='k', ls='--', lw=1.5, label='target 5%')
axes[2].set_xlabel('selection rate'); axes[2].set_ylabel('count')
axes[2].set_title('Selection rates (estimation stocks)'); axes[2].legend()

plt.tight_layout()
plt.savefig('estimation_results.png', dpi=150, bbox_inches='tight')
plt.show()

df_estim.to_csv('cross_sectional_results.csv', index=False)
print('Saved estimation_results.png, cross_sectional_results.csv')

## Rationale

**Why not tune on the EW aggregate?**  
Averaging 324 stocks cancels the idiosyncratic belief indices — the aggregate f_t ≈ 0
always, giving a completely different (and weaker) identification problem for κ.
Tuning on individual stocks directly gives the right signal.

**Why cross-sectional split instead of time split?**  
With optimal window s ~ 200–300 days, a 745-day hold-out contains only 2–3 non-overlapping
windows — too few to tune reliably. The cross-sectional split gives 1,879 days per stock
in both sets.

**Why Stage-1 R² excluded from objective?**  
Oracle test (Appendix) shows Stage-1 OOS R² ≈ 0 is expected at CMCE. In-sample R²
rewards in-window overfit.